# Calibration of D_geo Formula's terms

## Code to know camera resolution


In [2]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not opened")
    exit()

width  = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
fps    = cap.get(cv2.CAP_PROP_FPS)

print("Camera Resolution:", int(width), "x", int(height))
print("FPS:", fps)

cap.release()


Camera Resolution: 640 x 480
FPS: 30.0


## Lock the resolution

In [3]:
import cv2

cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

ret, frame = cap.read()

print("Actual Resolution:", frame.shape[1], "x", frame.shape[0])

cap.release()


Actual Resolution: 640 x 480


## Access your web camera

In [1]:
import cv2

cap = cv2.VideoCapture(0)

# Lock resolution (VERY IMPORTANT)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("Camera not opened")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Webcam", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Calculating HFOV

In [5]:
import math

D_wall = 1.76  # meters (your measured distance) ( distance from camera to wall)
W_wall = 1.785   # meters (replace with your measured width) ( width of the wall visible in the frame)

HFOV_rad = 2 * math.atan((W_wall / 2) / D_wall)
hFOV_deg = math.degrees(HFOV_rad)

print("Measured Horizontal FOV:", hFOV_deg, "degrees")


Measured Horizontal FOV: 53.77933936777884 degrees


## Compute focal length in PIXELS (fx)

In [6]:
import math

W_px = 640
HFOV_deg = hFOV_deg   # replace with your measured HFOV

HFOV_rad = math.radians(HFOV_deg)
fx = W_px / (2 * math.tan(HFOV_rad / 2))

print("Focal length fx:", fx, "pixels")


Focal length fx: 631.0364145658264 pixels


## finding the pixel of bbx

* Getting p value we will check using formula by knowing distance and width of the object and varify that our D_geo is working correctly (approx good or not)

In [7]:
from ultralytics import YOLO
import cv2

# Load YOLO model
yolo = YOLO(r"D:\IITBHU Internship\code\runs\detect\train11\best.pt")

# Read and resize image (MUST match calibration resolution)
img = cv2.imread(r"C:\Users\100ra\OneDrive\Pictures\Camera Roll\WIN_20251223_16_24_19_Pro.jpg")
img = cv2.resize(img, (640, 480))

# Run YOLO inference
results = yolo(img, conf=0.4, verbose=False)[0]

# Flag to check detection
detected = False

for box in results.boxes:
    detected = True

    # Bounding box coordinates
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    conf = float(box.conf[0])

    # Compute p (pixel size)
    bbox_w = x2 - x1
    bbox_h = y2 - y1
    p = max(bbox_w, bbox_h)

    print("YOLO p value:", p)

    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Put label text
    label = f"p = {p}px | conf = {conf:.2f}"
    cv2.putText(
        img,
        label,
        (x1, y1 - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

if not detected:
    print("No drone detected in image.")

# Show image with bounding box
cv2.imshow("YOLO p-value Calibration", img)
cv2.waitKey(0)
cv2.destroyAllWindows()



YOLO p value: 200


In [8]:
S = 0.5     # known real drone width (meters)

D_est = (S * fx) / p
print("Estimated distance:", D_est)


Estimated distance: 1.577591036414566


# Calibration of Depth map Formula's terms

In [10]:
import torch
import cv2
import numpy as np

class MidasDepth:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = torch.hub.load(
            "intel-isl/MiDaS", "MiDaS_small"
        ).to(self.device).eval()

        self.transform = torch.hub.load(
            "intel-isl/MiDaS", "transforms"
        ).small_transform

    def infer(self, frame):
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_batch = self.transform(img).to(self.device)

        with torch.no_grad():
            prediction = self.model(input_batch)
            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=img.shape[:2],
                mode="bicubic",
                align_corners=False
            ).squeeze()

        return prediction.cpu().numpy()


In [11]:
from ultralytics import YOLO
import cv2
import numpy as np

# ---------------- CONFIG ----------------
YOLO_MODEL_PATH = r"D:\IITBHU Internship\code\runs\detect\train11\best.pt"
CONF_THRESH = 0.4

# ---------------------------------------
yolo = YOLO(YOLO_MODEL_PATH)
depth_net = MidasDepth()

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

scale_values = []

print("Press 'c' to capture depth at known distance")
print("Press 'q' to finish calibration")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo(frame, conf=CONF_THRESH, verbose=False)[0]
    depth_map = depth_net.infer(frame)

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

        roi = depth_map[y1:y2, x1:x2]
        d_raw = np.median(roi)

        cv2.putText(frame, f"d_raw={d_raw:.2f}",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,0), 2)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):
            D_known = float(input("Enter known distance (meters): "))
            s = D_known * d_raw
            scale_values.append(s)
            print(f"Captured scale s = {s:.2f}")

    cv2.imshow("Depth Calibration", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("\nAll scale samples:", scale_values)
print("Final calibrated scale s =", np.mean(scale_values))


Using cache found in C:\Users\100ra/.cache\torch\hub\intel-isl_MiDaS_master
c:\Users\100ra\anaconda3\envs\ram_env\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


Using cache found in C:\Users\100ra/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\100ra/.cache\torch\hub\intel-isl_MiDaS_master


Press 'c' to capture depth at known distance
Press 'q' to finish calibration
Captured scale s = 627.14
Captured scale s = 799.94
Captured scale s = 879.50
Captured scale s = 779.97
Captured scale s = 662.75
Captured scale s = 604.28
Captured scale s = 583.70
Captured scale s = 355.11
Captured scale s = 250.46
Captured scale s = 1601.28

All scale samples: [np.float32(627.1447), np.float32(799.93665), np.float32(879.5002), np.float32(779.96655), np.float32(662.75146), np.float32(604.2778), np.float32(583.6969), np.float32(355.10815), np.float32(250.4646), np.float32(1601.2775)]
Final calibrated scale s = 714.4125


## Fused Geo + Depth

In [13]:
from ultralytics import YOLO
import cv2
import torch
import numpy as np

# ===================== USER PARAMETERS =====================
YOLO_MODEL_PATH = r"D:\IITBHU Internship\code\runs\detect\train11\best.pt"

S = 0.5          # assumed drone width (meters)
fx = 550.8       # calibrated focal length (pixels)
s = 680.0        # calibrated depth scale
CONF_THRESH = 0.4
P_REF = 120      # reference pixel size for geometry confidence

# ===================== LOAD MODELS =====================
yolo = YOLO(YOLO_MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small").to(device).eval()
transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

# ===================== CAMERA =====================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("✅ Fusion-based distance estimation running (press q to quit)")

# ===================== MAIN LOOP =====================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ---------- YOLO ----------
    yolo_res = yolo(frame, conf=CONF_THRESH, verbose=False)[0]

    # ---------- DEPTH ----------
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():
        depth = midas(input_batch)
        depth = torch.nn.functional.interpolate(
            depth.unsqueeze(1),
            size=frame.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze()

    depth_map = depth.cpu().numpy()

    for box in yolo_res.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        bbox_w = x2 - x1
        bbox_h = y2 - y1
        p = max(bbox_w, bbox_h)
        if p <= 0:
            continue

        # -------- Geometry distance --------
        D_geom = (S * fx) / p

        # -------- Depth distance --------
        roi = depth_map[y1:y2, x1:x2]
        d_raw = np.median(roi)
        D_depth = s / d_raw

        # -------- Confidences --------
        w_geom = min(1.0, p / P_REF)
        depth_std = np.std(roi)
        w_depth = 1.0 / (1.0 + depth_std)

        # Normalize
        w_sum = w_geom + w_depth
        w_geom /= w_sum
        w_depth /= w_sum

        # -------- Fusion --------
        D_fused = w_geom * D_geom + w_depth * D_depth

        # -------- Visualization --------
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        label = f"D={D_fused:.2f}m | G={D_geom:.2f} | Dp={D_depth:.2f}"
        cv2.putText(frame, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        print(f"Geom={D_geom:.2f} | Depth={D_depth:.2f} | Fused={D_fused:.2f}")

    cv2.imshow("Fusion Distance Estimation", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Using cache found in C:\Users\100ra/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


Using cache found in C:\Users\100ra/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\100ra/.cache\torch\hub\intel-isl_MiDaS_master


✅ Fusion-based distance estimation running (press q to quit)
Geom=1.62 | Depth=2.28 | Fused=1.62
Geom=1.72 | Depth=2.73 | Fused=1.73
Geom=1.73 | Depth=2.85 | Fused=1.74
Geom=1.80 | Depth=3.04 | Fused=1.81
Geom=1.85 | Depth=2.82 | Fused=1.85
Geom=1.84 | Depth=2.95 | Fused=1.84
Geom=1.81 | Depth=2.87 | Fused=1.82
Geom=1.86 | Depth=2.69 | Fused=1.87
Geom=1.75 | Depth=2.66 | Fused=1.76
Geom=1.75 | Depth=2.66 | Fused=1.76
Geom=1.77 | Depth=2.79 | Fused=1.77
Geom=1.77 | Depth=2.79 | Fused=1.77
Geom=1.78 | Depth=2.28 | Fused=1.78
Geom=1.79 | Depth=2.57 | Fused=1.79
Geom=1.79 | Depth=2.57 | Fused=1.79
Geom=1.77 | Depth=3.25 | Fused=1.77
Geom=1.73 | Depth=2.94 | Fused=1.74
Geom=1.73 | Depth=2.94 | Fused=1.74
Geom=2.62 | Depth=3.71 | Fused=2.63
Geom=2.62 | Depth=3.71 | Fused=2.63
Geom=1.75 | Depth=2.73 | Fused=1.76
Geom=1.71 | Depth=2.88 | Fused=1.72
Geom=1.71 | Depth=2.88 | Fused=1.72
Geom=1.70 | Depth=2.55 | Fused=1.70
Geom=1.70 | Depth=2.55 | Fused=1.70
Geom=1.67 | Depth=2.71 | Fused=1.67
Geo